**Conexão do Dataset do Roboflow**

In [ ]:
!pip install -q roboflow transformers datasets accelerate evaluate albumentations

from roboflow import Roboflow
from datasets import load_dataset

rf = Roboflow(api_key="SUA_API_KEY")
project = rf.workspace("seu-workspace").project("seu-projeto")
version = project.version(2) # Usando a v2 que você setou
dataset_roboflow = version.download("png-mask-semantic")

# Carregamento imediato
ds_hf = load_dataset("imagefolder", data_dir=dataset_roboflow.location)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 38.7 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to tcc---modelo-segformer-2 in png-mask-semantic:: 100%|██████████| 637/637 [00:00<00:00, 8140.71it/s]


Resolving data files:   0%|          | 0/553 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/53 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

**Treinamento SegFormer**

Preparação e IDs

In [ ]:
# Renomear para o padrão do SegFormer
train_ds = ds_hf["train"].rename_column("label", "labels")
if "validation" in ds_hf:
    test_ds = ds_hf["validation"].rename_column("label", "labels")
else:
    split = train_ds.train_test_split(test_size=0.1)
    train_ds, test_ds = split["train"], split["test"]

# Mapeamento de classes
id2label = {i: c for i, c in enumerate(project.classes)}
label2id = {c: i for i, c in enumerate(project.classes)}

Processador e Transformação

In [ ]:
import os
from PIL import Image
from datasets import Dataset
from transformers import SegformerImageProcessor

def load_roboflow_flat_dir(directory):
    all_files = sorted(os.listdir(directory))
    # No formato do Roboflow, as imagens são .jpg e as máscaras são .png
    # Elas geralmente têm o mesmo prefixo de nome
    images = [os.path.join(directory, f) for f in all_files if f.endswith('.jpg')]
    masks = [os.path.join(directory, f) for f in all_files if f.endswith('.png')]

    # Garante que temos o mesmo número de ambos
    if len(images) != len(masks):
        print(f"⚠️ Aviso em {directory}: {len(images)} imagens e {len(masks)} máscaras. Alinhando...")
        # Alinhamento básico pelo nome (caso falte algum par)
        actual_images, actual_masks = [], []
        mask_names = {os.path.basename(m).replace('.png', ''): m for m in masks}
        for img_path in images:
            base = os.path.basename(img_path).replace('.jpg', '')
            if base in mask_names:
                actual_images.append(img_path)
                actual_masks.append(mask_names[base])
        return Dataset.from_dict({"image": actual_images, "labels": actual_masks})

    return Dataset.from_dict({"image": images, "labels": masks})

# 1. Carregar as pastas que aparecem no seu print
base_path = "/content/tcc---modelo-segformer-2"
train_ds = load_roboflow_flat_dir(os.path.join(base_path, "train"))
test_ds = load_roboflow_flat_dir(os.path.join(base_path, "valid")) # Usando valid para teste real

# 2. Processador e Transformação
processor = SegformerImageProcessor.from_pretrained("nvidia/mit-b0")

def transform_fn(batch):
    # Abrindo os caminhos que guardamos no dataset
    images = [Image.open(x).convert("RGB") for x in batch["image"]]
    masks = [Image.open(x).convert("L") for x in batch["labels"]]
    return processor(images, segmentation_maps=masks, return_tensors="pt")

train_ds.set_transform(transform_fn)
test_ds.set_transform(transform_fn)

# 3. TESTE DE FOGO
try:
    item = train_ds[0]
    print(f"✅ SUCESSO! Dataset pronto com {len(train_ds)} amostras.")
    print(f"Formato dos Tensores: {item['pixel_values'].shape}")
except Exception as e:
    print(f"❌ Erro: {e}")


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'


✅ SUCESSO! Dataset pronto com 276 amostras.
Formato dos Tensores: torch.Size([3, 512, 512])


Modelo, Métricas e Treino

In [ ]:
import numpy as np
import evaluate
import torch
from transformers import (
    SegformerForSemanticSegmentation,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

# 1. Configurações Básicas (Garante que tudo existe na memória)
id2label = {i: c for i, c in enumerate(project.classes)}
label2id = {c: i for i, c in enumerate(project.classes)}

# 2. Carregar o Modelo
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=len(project.classes),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

metric = evaluate.load("mean_iou")

# 3. Definição dos Argumentos de Treino (O que estava faltando!)
training_args = TrainingArguments(
    output_dir="segformer-tcc-armas",
    learning_rate=6e-5,
    num_train_epochs=50,
    per_device_train_batch_size=4,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    remove_unused_columns=False,
)

# 4. Função de métricas (com a correção do erro de JSON)
def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        logits = torch.from_numpy(logits)

        logits = torch.nn.functional.interpolate(
            logits, size=labels.shape[-2:], mode="bilinear", align_corners=False
        ).argmax(dim=1)

        pred_labels = logits.detach().cpu().numpy()

        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=len(project.classes),
            ignore_index=255,
        )

        # Limpeza para evitar o erro de JSON serializable
        metrics.pop("per_category_accuracy", None)
        metrics.pop("per_category_iou", None)

        return {k: float(v) if isinstance(v, np.ndarray) else v for k, v in metrics.items()}

# 5. Criar o Trainer e rodar
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    data_collator=DefaultDataCollator(),
)

print("🚀 Retomando treinamento oficial do TCC...")
trainer.train()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.bias                               | UNEXPECTED | 
classifier.weight                             | UNEXPECTED | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

...

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.


Step,Training Loss,Validation Loss,Mean Iou,Mean Accuracy,Overall Accuracy
100,No log,0.606933,0.469323,0.580025,0.976018


Downcasting array dtype int64 to int32 to be compatible with 'Pillow'


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.


Pegar as métricas que faltam (Latência e FPS)

In [ ]:
import time
import torch

# 1. Preparar o modelo para teste
model.eval()
model.to("cuda")

# 2. Criar um dado fictício (mesmo tamanho do SegFormer MiT-B0)
dummy_input = torch.randn(1, 3, 512, 512).to("cuda")

# 3. Warm-up (descarta as primeiras execuções que são lentas)
for _ in range(20):
    _ = model(dummy_input)

# 4. Medição Real
starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
repetitions = 100
timings = np.zeros((repetitions, 1))

with torch.no_grad():
    for rep in range(repetitions):
        starter.record()
        _ = model(dummy_input)
        ender.record()
        torch.cuda.synchronize()
        curr_time = starter.elapsed_time(ender)
        timings[rep] = curr_time

avg_latencia = np.sum(timings) / repetitions
fps = 1000 / avg_latencia

print(f"⏱️ Latência Média: {avg_latencia:.2f} ms")
print(f"🚀 FPS: {fps:.2f}")

# **Código Completo**

In [ ]:
!pip install -q roboflow transformers datasets accelerate evaluate albumentations

from roboflow import Roboflow
from datasets import load_dataset

rf = Roboflow(api_key="SUA_API_KEY")
project = rf.workspace("seu-workspace").project("seu-projeto")
version = project.version(2) # Usando a v2 que você setou
dataset_roboflow = version.download("png-mask-semantic")

# Carregamento imediato
ds_hf = load_dataset("imagefolder", data_dir=dataset_roboflow.location)




# Renomear para o padrão do SegFormer
train_ds = ds_hf["train"].rename_column("label", "labels")
if "validation" in ds_hf:
    test_ds = ds_hf["validation"].rename_column("label", "labels")
else:
    split = train_ds.train_test_split(test_size=0.1)
    train_ds, test_ds = split["train"], split["test"]

# Mapeamento de classes
id2label = {i: c for i, c in enumerate(project.classes)}
label2id = {c: i for i, c in enumerate(project.classes)}





import os
from PIL import Image
from datasets import Dataset
from transformers import SegformerImageProcessor

def load_roboflow_flat_dir(directory):
    all_files = sorted(os.listdir(directory))
    # No formato do Roboflow, as imagens são .jpg e as máscaras são .png
    # Elas geralmente têm o mesmo prefixo de nome
    images = [os.path.join(directory, f) for f in all_files if f.endswith('.jpg')]
    masks = [os.path.join(directory, f) for f in all_files if f.endswith('.png')]

    # Garante que temos o mesmo número de ambos
    if len(images) != len(masks):
        print(f"⚠️ Aviso em {directory}: {len(images)} imagens e {len(masks)} máscaras. Alinhando...")
        # Alinhamento básico pelo nome (caso falte algum par)
        actual_images, actual_masks = [], []
        mask_names = {os.path.basename(m).replace('.png', ''): m for m in masks}
        for img_path in images:
            base = os.path.basename(img_path).replace('.jpg', '')
            if base in mask_names:
                actual_images.append(img_path)
                actual_masks.append(mask_names[base])
        return Dataset.from_dict({"image": actual_images, "labels": actual_masks})

    return Dataset.from_dict({"image": images, "labels": masks})

# 1. Carregar as pastas que aparecem no seu print
base_path = "/content/tcc---modelo-segformer-2"
train_ds = load_roboflow_flat_dir(os.path.join(base_path, "train"))
test_ds = load_roboflow_flat_dir(os.path.join(base_path, "valid")) # Usando valid para teste real

# 2. Processador e Transformação
processor = SegformerImageProcessor.from_pretrained("nvidia/mit-b0")

def transform_fn(batch):
    # Abrindo os caminhos que guardamos no dataset
    images = [Image.open(x).convert("RGB") for x in batch["image"]]
    masks = [Image.open(x).convert("L") for x in batch["labels"]]
    return processor(images, segmentation_maps=masks, return_tensors="pt")

train_ds.set_transform(transform_fn)
test_ds.set_transform(transform_fn)

# 3. TESTE DE FOGO
try:
    item = train_ds[0]
    print(f"✅ SUCESSO! Dataset pronto com {len(train_ds)} amostras.")
    print(f"Formato dos Tensores: {item['pixel_values'].shape}")
except Exception as e:
    print(f"❌ Erro: {e}")






import numpy as np
import evaluate
import torch
from transformers import (
    SegformerForSemanticSegmentation,
    TrainingArguments,
    Trainer,
    DefaultDataCollator
)

# 1. Configurações Básicas (Garante que tudo existe na memória)
id2label = {i: c for i, c in enumerate(project.classes)}
label2id = {c: i for i, c in enumerate(project.classes)}

# 2. Carregar o Modelo
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=len(project.classes),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

metric = evaluate.load("mean_iou")

# 3. Definição dos Argumentos de Treino (O que estava faltando!)
training_args = TrainingArguments(
    output_dir="segformer-tcc-armas",
    learning_rate=6e-5,
    num_train_epochs=50,
    per_device_train_batch_size=4,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    remove_unused_columns=False,
)

# 4. Função de métricas (com a correção do erro de JSON)
def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        logits = torch.from_numpy(logits)

        logits = torch.nn.functional.interpolate(
            logits, size=labels.shape[-2:], mode="bilinear", align_corners=False
        ).argmax(dim=1)

        pred_labels = logits.detach().cpu().numpy()

        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=len(project.classes),
            ignore_index=255,
        )

        # Limpeza para evitar o erro de JSON serializable
        metrics.pop("per_category_accuracy", None)
        metrics.pop("per_category_iou", None)

        return {k: float(v) if isinstance(v, np.ndarray) else v for k, v in metrics.items()}

# 5. Criar o Trainer e rodar
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    data_collator=DefaultDataCollator(),
)

print("🚀 Retomando treinamento oficial do TCC...")
trainer.train()